# FieldGuide sample ingestion
Run on Databricks serverless. Use only the public sample export. No Gemini API or additional packages are required.

Run the setup cell first, upload `sample_pages.jsonl` to the printed volume path, then run the remaining cells. Change CATALOG if your writable catalog has a different name. Each run creates its own output table and file.


In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
from pyspark.sql import functions as F

CATALOG = spark.catalog.currentCatalog()
SCHEMA = "fieldguide"
VOLUME = "demo_files"
CHUNK_SIZE = 1000
OVERLAP = 150
assert CHUNK_SIZE >= 100 and 0 <= OVERLAP < CHUNK_SIZE

def identifier(value):
    return "`" + value.replace("`", "``") + "`"

namespace = f"{identifier(CATALOG)}.{identifier(SCHEMA)}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {namespace}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {namespace}.{identifier(VOLUME)}")
volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
print(f"Upload sample_pages.jsonl here: {volume_path}")


Upload the local `reports/databricks/sample_pages.jsonl` file using **Catalog ? your catalog ? fieldguide ? demo_files ? Upload**. Then continue below.


In [ ]:
pages = (spark.read.schema("source STRING, page INT, text STRING")
         .option("mode", "FAILFAST")
         .json(f"{volume_path}/sample_pages.jsonl"))
invalid = pages.filter(
    F.col("source").isNull() | (F.length(F.trim("source")) == 0)
    | F.col("text").isNull() | (F.length(F.trim("text")) == 0)
    | (F.col("page").isNotNull() & (F.col("page") < 1))
)
assert invalid.limit(1).count() == 0, "Input has blank text or invalid source metadata"
assert pages.limit(1).count() > 0, "No pages found"
assert pages.groupBy("source", "page").count().filter("count > 1").limit(1).count() == 0, "Duplicate pages"
display(pages.select("source", "page", F.length("text").alias("characters")))


In [ ]:
step = CHUNK_SIZE - OVERLAP
chunks = (pages
    .withColumn("start_index", F.explode(F.sequence(
        F.lit(0), F.greatest(F.length("text") - F.lit(OVERLAP + 1), F.lit(0)), F.lit(step))))
    .withColumn("text", F.expr(f"substring(text, start_index + 1, {CHUNK_SIZE})"))
    .filter(F.length(F.trim("text")) > 0)
    .withColumn("chunk_size", F.lit(CHUNK_SIZE))
    .withColumn("chunk_overlap", F.lit(OVERLAP)))
identity = F.concat(F.col("source"), F.lit(":"),
                    F.coalesce(F.col("page").cast("string"), F.lit("None")), F.lit(":"),
                    F.col("start_index").cast("string"), F.lit(":"), F.col("text"))
chunks = chunks.withColumn("chunk_id", F.substring(F.sha2(identity, 256), 1, 16))
assert chunks.groupBy("chunk_id").count().filter("count > 1").limit(1).count() == 0
assert chunks.filter(F.length("text") > CHUNK_SIZE).limit(1).count() == 0
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
table_name = f"{namespace}.chunks_{run_id}"
chunks.write.format("delta").mode("errorifexists").saveAsTable(table_name)
stored = spark.table(table_name)
display(stored.groupBy("source").agg(F.count("*").alias("chunks")))
print(f"Saved {stored.count()} chunks in {table_name}")


The Spark path uses fixed character windows; the default local ingester uses paragraph-aware splitting. Chunk boundaries can differ. Both preserve source locations.

The following single-file export is intended for this small demo. Larger corpora should use partitioned Spark output instead of routing all rows through the notebook driver.


In [ ]:
output = Path(volume_path) / f"chunks_{run_id}.jsonl"
with output.open("x", encoding="utf-8") as handle:
    for row in stored.orderBy("source", "page", "start_index").toLocalIterator():
        handle.write(json.dumps(row.asDict(), ensure_ascii=False) + "\n")
print(f"Download this file from Catalog: {output}")
